# Sesión 04 – HDFS, Formatos y Diseño de Almacenamiento aplicado al Proyecto de Ventas

**Objetivo:** Diseñar el almacenamiento del pipeline batch usando **formato columnar Parquet con particionado**, aplicado al dataset real de facturas electrónicas 2021-2026. Entender por qué el diseño del almacenamiento impacta directamente el rendimiento de las consultas analíticas.

**Dataset:** `Ventas_2021-2026.xlsx` — facturas electrónicas del negocio (retomamos desde el ETL del NB03).

---

## Contexto: ¿Qué es HDFS y por qué importa?

**HDFS** (Hadoop Distributed File System) es el sistema de archivos distribuido de Hadoop. En lugar de guardar un archivo en un solo disco, HDFS lo **divide en bloques** y los distribuye entre múltiples máquinas.

```
┌─────────────────────────────────────────────────────────────────┐
│                    ARQUITECTURA HDFS                             │
│                                                                  │
│  ┌────────────┐        ┌──────────────────────────────────┐     │
│  │ NameNode   │        │          DataNodes                │     │
│  │            │        │                                  │     │
│  │ Directorio │───────▶│ DN1: bloque1, bloque4, bloque7   │     │
│  │ de bloques │        │ DN2: bloque2, bloque5, bloque8   │     │
│  │ (metadata) │        │ DN3: bloque3, bloque6, bloque9   │     │
│  └────────────┘        └──────────────────────────────────┘     │
│                                                                  │
│  🔑 NameNode: sabe DÓNDE está cada bloque                       │
│  💾 DataNode: guarda los bloques realmente                       │
└─────────────────────────────────────────────────────────────────┘
```

> **Nota del laboratorio:** En esta sesión NO tenemos un HDFS real con NameNode y DataNodes. Trabajamos con el **sistema de archivos local** simulando la misma lógica: leer desde una zona de almacenamiento → procesar con Spark → persistir salida optimizada.

## Zonas de un Data Lake (arquitectura real)

| Zona | Nombre | Contenido | Formato típico |
|---|---|---|---|
| **Raw** (Bronce) | Zona de aterrizaje | Datos originales sin tocar | Excel, CSV, JSON |
| **Processed** (Plata) | Zona limpia | Datos limpios y tipados | Parquet sin partición |
| **Analytics** (Oro) | Zona analítica | Agregaciones y salidas ETL | Parquet particionado |

```
En nuestro proyecto:

📁 data/Ventas_2021-2026.xlsx    ← Zona RAW (Excel original)
         │
         ▼ ETL (NB03)
📁 artifacts/output_etl_ventas/  ← Zona PROCESSED (Parquet limpio)
         │
         ▼ Particionado (este NB)
📁 artifacts/ventas_particionado/ ← Zona ANALYTICS (Parquet particionado por anio/mes)
```

**Temas de esta sesión:**
- Leer el Parquet del NB03 (zona Processed)
- Revisar calidad antes de almacenar
- ¿Qué es el particionado y por qué importa?
- Escribir Parquet particionado por `anio` y `mes`
- Explorar la estructura física de carpetas
- Consulta con Partition Pruning (Spark lee solo lo necesario)
- Comparación CSV vs Parquet sin partición vs Parquet particionado
- Diseño de almacenamiento: decisiones técnicas justificadas

## Paso 1 — Inicializar SparkSession

Configuración estándar del proyecto. Usamos `local[*]` para usar todos los núcleos disponibles.

In [2]:
import pandas as pd
import warnings
import unicodedata
import os
import shutil
warnings.filterwarnings('ignore')

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, to_date, broadcast, when, year, month, quarter, date_format
from pyspark.sql.functions import sum as _sum, row_number, count, avg, min as _min, max as _max
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("ProyectoVentas_HDFS_Formatos")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"✅ SparkSession lista")
print(f"   Versión  : {spark.version}")
print(f"   App Name : {spark.sparkContext.appName}")
print(f"   Master   : {spark.sparkContext.master}")
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/14 13:41:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/14 13:41:34 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/14 13:41:34 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/14 13:41:34 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


✅ SparkSession lista
   Versión  : 4.1.1
   App Name : ProyectoVentas_HDFS_Formatos
   Master   : local[*]


## Paso 2 — Leer el dataset (Zona RAW → Zona Processed)

Intentamos primero leer el Parquet que generamos en el NB03 (`artifacts/output_etl_ventas`). Si no existe (primera ejecución), ejecutamos nuevamente el pipeline de ingesta completo desde el Excel.

Esto refleja una práctica real: **siempre que sea posible, reutilizar datos ya procesados** en lugar de rehacer el ETL completo.

In [3]:
# ── Función para normalizar nombres de columnas ──
def normalizar_columna(nombre):
    nombre = str(nombre).strip().lower()
    nombre = unicodedata.normalize('NFD', nombre)
    nombre = ''.join(c for c in nombre if unicodedata.category(c) != 'Mn')
    nombre = nombre.replace(' ', '_').replace('/', '_').replace('-', '_')
    return nombre

# ── Rutas del proyecto ──
EXCEL_PATH          = "../data/Ventas_2021-2026.xlsx"
PARQUET_NB03_PATH   = "../artifacts/output_etl_ventas"   # salida del NB03
RUTA_BASE           = "../artifacts/nb04_almacenamiento"  # carpeta base de esta sesión

RUTA_CSV              = f"{RUTA_BASE}/ventas_csv"
RUTA_PARQUET_SIMPLE   = f"{RUTA_BASE}/ventas_parquet_simple"
RUTA_PARQUET_PART_ANO = f"{RUTA_BASE}/ventas_parquet_por_anio"
RUTA_PARQUET_PART_MES = f"{RUTA_BASE}/ventas_parquet_por_anio_mes"

# ── Intentar leer Parquet del NB03 primero ──
parquet_nb03_existe = os.path.exists(PARQUET_NB03_PATH)

if parquet_nb03_existe:
    print(f"✅ Parquet del NB03 encontrado → leyendo desde zona Processed")
    df_ventas = spark.read.parquet(PARQUET_NB03_PATH)
    print(f"   Registros : {df_ventas.count():,}")
    print(f"   Columnas  : {len(df_ventas.columns)}")
else:
    print(f"⚠️  Parquet del NB03 NO encontrado → ejecutando pipeline desde Excel (zona RAW)")
    print(f"   Ruta buscada: {os.path.abspath(PARQUET_NB03_PATH)}")

✅ Parquet del NB03 encontrado → leyendo desde zona Processed
   Registros : 522,688
   Columnas  : 32


In [9]:
# ── Si no existe el Parquet del NB03, hacemos el pipeline desde cero ──
if not parquet_nb03_existe:
    print("⏳ Cargando Excel con pandas...")
    df_pd = pd.read_excel(EXCEL_PATH, dtype=str)
    df_pd.columns = [normalizar_columna(c) for c in df_pd.columns]

    ventas_raw = spark.createDataFrame(df_pd)

    # Detectar columnas
    col_fecha   = next((c for c in ventas_raw.columns if 'fecha' in c), None)
    col_total   = next((c for c in reversed(ventas_raw.columns) if 'total' in c), None)
    col_anulado = next((c for c in ventas_raw.columns if 'anulado' in c), None)
    col_doc     = next((c for c in ventas_raw.columns if 'numero' in c and 'entidad' in c), None)
    col_cliente = next((c for c in ventas_raw.columns if 'denominacion' in c or 'entidad' in c), None)

    # Pipeline ETL simplificado
    df_ventas = ventas_raw
    if col_total:
        df_ventas = df_ventas.withColumn(col_total, F.regexp_replace(F.col(col_total), ',', '.').cast('double'))
    if col_fecha:
        df_ventas = df_ventas.withColumn('FECHA_DT', F.to_date(F.col(col_fecha), 'dd/MM/yyyy'))
    cols_nn = [c for c in [col_doc, col_fecha, col_total] if c]
    df_ventas = df_ventas.dropna(subset=cols_nn)
    if col_anulado:
        df_ventas = df_ventas.filter(
            (~F.upper(F.col(col_anulado)).isin('SI','S','1','TRUE','ANULADO')) &
            (F.col(col_total) > 0)
        )

    print(f"✅ Pipeline ejecutado desde Excel → {df_ventas.count():,} registros limpios")

# Detectar columnas clave (sea cual sea la fuente)
FECHA_COL   = next((c for c in df_ventas.columns if 'FECHA_DT' in c or 'fecha' in c.lower()), None)
TOTAL_COL   = next((c for c in reversed(df_ventas.columns) if 'total' in c.lower()), None)
DOC_COL     = next((c for c in df_ventas.columns if 'cliente_id' in c.lower() or ('numero' in c.lower() and 'entidad' in c.lower())), None)
CLIENTE_COL = next((c for c in df_ventas.columns if 'nombre_cliente' in c.lower() or 'denominacion' in c.lower()), None)

print(f"\n🔍 Columnas clave detectadas:")
print(f"   Fecha   : {FECHA_COL}")
print(f"   Total   : {TOTAL_COL}")
print(f"   Doc/ID  : {DOC_COL}")
print(f"   Cliente : {CLIENTE_COL}")


🔍 Columnas clave detectadas:
   Fecha   : fecha_de_emision
   Total   : total_acumulado
   Doc/ID  : doc_entidad_numero
   Cliente : denominacion_entidad


## Paso 3 — Revisar la estructura del dataset

Antes de diseñar el almacenamiento, revisamos:
1. Qué columnas tenemos disponibles
2. Los tipos de datos actuales
3. El volumen y cobertura temporal del dataset

> **Pregunta clave de ingeniería:** ¿Estoy leyendo el dato con la estructura esperada antes de diseñar la salida?
> Si el esquema viene mal desde el inicio, el problema se arrastra al particionado y al almacenamiento final.

In [10]:
# ── Explorar estructura ──
print("📋 Schema del dataset:")
df_ventas.printSchema()

# Estadísticas básicas
n_registros  = df_ventas.count()
n_columnas   = len(df_ventas.columns)
n_particiones = df_ventas.rdd.getNumPartitions()

print(f"\n📊 Estadísticas del dataset:")
print(f"   Total registros     : {n_registros:,}")
print(f"   Total columnas      : {n_columnas}")
print(f"   Particiones Spark   : {n_particiones}")

# Rango temporal
if FECHA_COL and FECHA_COL in df_ventas.columns:
    rango = df_ventas.agg(
        _min(FECHA_COL).alias("desde"),
        _max(FECHA_COL).alias("hasta")
    ).collect()[0]
    print(f"   Rango de fechas     : {rango['desde']} → {rango['hasta']}")

# Total de ventas
if TOTAL_COL and TOTAL_COL in df_ventas.columns:
    totales = df_ventas.agg(
        _sum(TOTAL_COL).alias("total_ventas"),
        avg(TOTAL_COL).alias("promedio")
    ).collect()[0]
    print(f"   Total ventas (S/)   : {totales['total_ventas']:,.2f}")
    print(f"   Promedio por línea  : {totales['promedio']:,.2f}")

print()
print("📄 Primeras 5 filas:")
df_ventas.show(5, truncate=True)

📋 Schema del dataset:
root
 |-- fecha_de_emision: string (nullable = true)
 |-- tipo: string (nullable = true)
 |-- serie: string (nullable = true)
 |-- numero: string (nullable = true)
 |-- doc_entidad_tipo: string (nullable = true)
 |-- doc_entidad_numero: string (nullable = true)
 |-- denominacion_entidad: string (nullable = true)
 |-- moneda: string (nullable = true)
 |-- tipo_de_cambio: string (nullable = true)
 |-- unidad_de_medida: string (nullable = true)
 |-- codigo: string (nullable = true)
 |-- descripcion: string (nullable = true)
 |-- cantidad: string (nullable = true)
 |-- valor_unitario: string (nullable = true)
 |-- precio_unitario: string (nullable = true)
 |-- descuento: string (nullable = true)
 |-- subtotal: string (nullable = true)
 |-- tipo_de_igv: string (nullable = true)
 |-- igv: string (nullable = true)
 |-- tipo_de_isc: double (nullable = true)
 |-- isc: string (nullable = true)
 |-- impuesto_bolsas: string (nullable = true)
 |-- total: double (nullable = tru

## Paso 4 — Preparar columnas para el particionado

Para almacenar con criterio, primero derivamos las **columnas de partición**: `anio`, `mes` y `trimestre`.

### ¿Por qué estas columnas?

Las preguntas más frecuentes en un negocio de ventas son por **tiempo**:
- "¿Cómo fueron las ventas de enero 2024?"
- "¿Cuánto vendimos en el Q3 de 2023?"
- "¿Cuál fue el mejor mes del año?"

Si particionamos por `anio` y `mes`, Spark puede leer **solo las carpetas relevantes** para esas preguntas.

In [11]:
# ── Preparar columnas de partición ──
df_preparado = df_ventas

# Aseguramos que la fecha esté como DateType
if FECHA_COL and FECHA_COL in df_preparado.columns:
    # Si ya es DateType, no hace falta reconvertir
    from pyspark.sql.types import DateType
    fecha_type = df_preparado.schema[FECHA_COL].dataType
    if not isinstance(fecha_type, DateType):
        df_preparado = df_preparado.withColumn(
            FECHA_COL,
            F.to_date(F.col(FECHA_COL), "dd/MM/yyyy")
        )

    # Crear columnas de partición
    df_preparado = (
        df_preparado
        .withColumn("anio",       year(col(FECHA_COL)))
        .withColumn("mes",        month(col(FECHA_COL)))
        .withColumn("trimestre",  quarter(col(FECHA_COL)))
        .withColumn("anio_mes",   date_format(col(FECHA_COL), "yyyy-MM"))
    )

    print("✅ Columnas de partición creadas:")
    print("   - anio      : año de la factura (ej: 2022, 2023, 2024)")
    print("   - mes       : mes numérico (1-12)")
    print("   - trimestre : trimestre (1-4)")
    print("   - anio_mes  : formato 'yyyy-MM' para reportes")
    print()

    # Ver distribución por año
    print("📊 Distribución de registros por año:")
    df_preparado.filter(col("anio").isNotNull()) \
        .groupBy("anio") \
        .agg(
            count("*").alias("num_registros"),
            _sum(TOTAL_COL).alias("total_ventas_soles")
        ) \
        .orderBy("anio") \
        .show()
else:
    print("⚠️  No se encontró columna de fecha para crear columnas de partición")

✅ Columnas de partición creadas:
   - anio      : año de la factura (ej: 2022, 2023, 2024)
   - mes       : mes numérico (1-12)
   - trimestre : trimestre (1-4)
   - anio_mes  : formato 'yyyy-MM' para reportes

📊 Distribución de registros por año:
+----+-------------+--------------------+
|anio|num_registros|  total_ventas_soles|
+----+-------------+--------------------+
|2021|       126411| 7.621456274403208E7|
|2022|       122011| 1.673082071771409E8|
|2023|        98207| 5.184849068916187E9|
|2024|        98740|2.4685391245373354E9|
|2025|        72328|2.907956784447483...|
|2026|         4991|1.025200153988480...|
+----+-------------+--------------------+



In [12]:
# ── Vista previa del DataFrame preparado para almacenamiento ──
print("📋 Schema final del DataFrame preparado:")
df_preparado.printSchema()

print("\n📄 Muestra del DataFrame preparado:")
cols_mostrar = [c for c in [FECHA_COL, TOTAL_COL, "anio", "mes", "trimestre"] if c in df_preparado.columns]
df_preparado.select(*cols_mostrar).show(8)

📋 Schema final del DataFrame preparado:
root
 |-- fecha_de_emision: date (nullable = true)
 |-- tipo: string (nullable = true)
 |-- serie: string (nullable = true)
 |-- numero: string (nullable = true)
 |-- doc_entidad_tipo: string (nullable = true)
 |-- doc_entidad_numero: string (nullable = true)
 |-- denominacion_entidad: string (nullable = true)
 |-- moneda: string (nullable = true)
 |-- tipo_de_cambio: string (nullable = true)
 |-- unidad_de_medida: string (nullable = true)
 |-- codigo: string (nullable = true)
 |-- descripcion: string (nullable = true)
 |-- cantidad: string (nullable = true)
 |-- valor_unitario: string (nullable = true)
 |-- precio_unitario: string (nullable = true)
 |-- descuento: string (nullable = true)
 |-- subtotal: string (nullable = true)
 |-- tipo_de_igv: string (nullable = true)
 |-- igv: string (nullable = true)
 |-- tipo_de_isc: double (nullable = true)
 |-- isc: string (nullable = true)
 |-- impuesto_bolsas: string (nullable = true)
 |-- total: double

## Paso 5 — Revisión de calidad antes de almacenar

**Regla de oro del almacenamiento:** No guardes basura — ¡la basura se queda para siempre!

Revisamos los problemas más comunes antes de escribir la salida final:
1. Nulos en columnas críticas
2. Totales negativos o cero
3. Registros con fechas inválidas (fuera del rango esperado 2021-2026)

In [13]:
# ── Validación 1: Nulos en columnas críticas ──
cols_validar = [c for c in [FECHA_COL, TOTAL_COL, "anio", "mes"] if c in df_preparado.columns]
print("🔍 Validación 1 — Nulos por columna (deben ser 0 para almacenar):")
df_preparado.select([
    count(F.when(col(c).isNull(), c)).alias(f"nulos_{c}")
    for c in cols_validar
]).show(truncate=False)

# ── Validación 2: Totales inválidos ──
if TOTAL_COL in df_preparado.columns:
    n_negativos = df_preparado.filter(col(TOTAL_COL) <= 0).count()
    print(f"🔍 Validación 2 — Registros con total <= 0: {n_negativos:,}")

# ── Validación 3: Fechas fuera de rango ──
if FECHA_COL in df_preparado.columns and "anio" in df_preparado.columns:
    fuera_rango = df_preparado.filter(
        (col("anio") < 2021) | (col("anio") > 2026)
    ).count()
    print(f"🔍 Validación 3 — Registros fuera del rango 2021-2026: {fuera_rango:,}")

print()
print(f"✅ Dataset listo para almacenamiento: {df_preparado.count():,} registros")

🔍 Validación 1 — Nulos por columna (deben ser 0 para almacenar):
+----------------------+---------------------+----------+---------+
|nulos_fecha_de_emision|nulos_total_acumulado|nulos_anio|nulos_mes|
+----------------------+---------------------+----------+---------+
|0                     |0                    |0         |0        |
+----------------------+---------------------+----------+---------+

🔍 Validación 2 — Registros con total <= 0: 0
🔍 Validación 3 — Registros fuera del rango 2021-2026: 0

✅ Dataset listo para almacenamiento: 522,688 registros


## Paso 6 — Pensar como ingenieros de datos: ¿Qué diseño de almacenamiento elegir?

La pregunta ya no es solo:
> "¿cómo guardo el DataFrame?"

La pregunta correcta es:
> "¿cómo diseño el almacenamiento para que futuras consultas lean **menos datos** y respondan **más rápido**?"

### Decisiones a tomar:

**1. Formato:**

| Formato | Compresión | Tipos | Lectura columnar | Uso ideal |
|---|---|---|---|---|
| **CSV** | ❌ No | ❌ Todo texto | ❌ No | Intercambio, compatibilidad |
| **Parquet** | ✅ Snappy/Gzip | ✅ Sí | ✅ Sí | Analítica batch, Data Lake |

**2. Particionado:**

El particionado divide el Parquet en **subcarpetas** según el valor de una columna. Spark lee solo la carpeta que necesita.

```
Sin partición:                   Con partición por anio/mes:
ventas_parquet/                  ventas_parquet/
  ├── part-00000.parquet  ← lee  ├── anio=2022/
  ├── part-00001.parquet    TODO │   ├── mes=1/ ← ¡solo lee esto!
  └── part-00002.parquet         │   └── mes=2/
                                 ├── anio=2023/
                                 └── anio=2024/
```

### ¿Cuándo el particionado es contraproducente?
- Si la columna de partición tiene **cardinalidad muy alta** (ej: cliente_id con miles de valores → miles de carpetas)
- Si las consultas raramente filtran por esa columna
- Si el dataset es muy pequeño (el overhead de metadatos es mayor que el beneficio)

## Paso 7 — Comparación de estrategias de almacenamiento

Guardamos el mismo dataset en **3 formatos/estrategias** para comparar:

1. **CSV** — formato texto plano (referencia)
2. **Parquet simple** — sin particioanr
3. **Parquet particionado por `anio` y `mes`** — diseño analítico óptimo para nuestro caso

In [14]:
import shutil

# Limpiar directorio base si existe
shutil.rmtree(RUTA_BASE, ignore_errors=True)
print(f"🗑️  Directorio anterior eliminado (si existía)")
print()

# ── Opción 1: CSV ──
print("💾 Guardando en CSV...")
df_preparado.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(RUTA_CSV)
print(f"   ✅ CSV guardado en: {RUTA_CSV}")

# ── Opción 2: Parquet simple (sin partición) ──
print()
print("💾 Guardando en Parquet simple (sin partición)...")
df_preparado.write \
    .mode("overwrite") \
    .parquet(RUTA_PARQUET_SIMPLE)
print(f"   ✅ Parquet simple guardado en: {RUTA_PARQUET_SIMPLE}")

# ── Opción 3: Parquet particionado por anio y mes ──
print()
print("💾 Guardando en Parquet particionado por anio y mes...")
print("   (esto crea subcarpetas: anio=2022/mes=1/, anio=2022/mes=2/, ...)")
df_preparado.write \
    .mode("overwrite") \
    .partitionBy("anio", "mes") \
    .parquet(RUTA_PARQUET_PART_MES)
print(f"   ✅ Parquet particionado guardado en: {RUTA_PARQUET_PART_MES}")

print()
print("✅ Las 3 estrategias de almacenamiento generadas correctamente")

🗑️  Directorio anterior eliminado (si existía)

💾 Guardando en CSV...


   ✅ CSV guardado en: ../artifacts/nb04_almacenamiento/ventas_csv

💾 Guardando en Parquet simple (sin partición)...


26/04/14 13:45:41 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

   ✅ Parquet simple guardado en: ../artifacts/nb04_almacenamiento/ventas_parquet_simple

💾 Guardando en Parquet particionado por anio y mes...
   (esto crea subcarpetas: anio=2022/mes=1/, anio=2022/mes=2/, ...)


26/04/14 13:45:43 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/14 13:45:43 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/14 13:45:43 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/14 13:45:43 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/14 13:45:43 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/14 13:45:43 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/14 13:45:43 WARN MemoryManager: Total allocation exceeds 95.00% 

   ✅ Parquet particionado guardado en: ../artifacts/nb04_almacenamiento/ventas_parquet_por_anio_mes

✅ Las 3 estrategias de almacenamiento generadas correctamente


## Paso 8 — Explorar la estructura física de carpetas

El **particionado** convierte el concepto lógico de `partitionBy()` en una **estructura real de carpetas** en disco. Esto es exactamente lo que ocurre en HDFS: cada partición se almacena en un bloque diferente.

Inspeccionemos la diferencia entre el Parquet simple y el particionado.

In [15]:
def mostrar_estructura(ruta, max_niveles=4, max_archivos=5):
    """Muestra la estructura de carpetas de forma legible."""
    if not os.path.exists(ruta):
        print(f"   ⚠️  No encontrado: {ruta}")
        return
    for root, dirs, files in os.walk(ruta):
        nivel = root.replace(ruta, "").count(os.sep)
        if nivel > max_niveles:
            continue
        sangria = "   " * nivel
        nombre = os.path.basename(root) if os.path.basename(root) else os.path.basename(ruta)
        print(f"{sangria}📁 {nombre}/")
        for i, f in enumerate(sorted(files)):
            if i >= max_archivos:
                print(f"{sangria}   ... ({len(files) - max_archivos} archivos más)")
                break
            icono = "📄" if f.endswith(".parquet") else "📃" if f.endswith(".csv") else "✅" if "_SUCCESS" in f else "📎"
            print(f"{sangria}   {icono} {f}")

print("═" * 60)
print("📂 PARQUET SIMPLE (sin partición):")
print("═" * 60)
mostrar_estructura(RUTA_PARQUET_SIMPLE, max_niveles=1)

print()
print("═" * 60)
print("📂 PARQUET PARTICIONADO (por anio / mes):")
print("═" * 60)
mostrar_estructura(RUTA_PARQUET_PART_MES, max_niveles=3, max_archivos=2)

════════════════════════════════════════════════════════════
📂 PARQUET SIMPLE (sin partición):
════════════════════════════════════════════════════════════
📁 ventas_parquet_simple/
   ✅ ._SUCCESS.crc
   📎 .part-00000-3ec73052-68ac-445c-803f-c54b87e9f16d-c000.snappy.parquet.crc
   📎 .part-00001-3ec73052-68ac-445c-803f-c54b87e9f16d-c000.snappy.parquet.crc
   📎 .part-00002-3ec73052-68ac-445c-803f-c54b87e9f16d-c000.snappy.parquet.crc
   📎 .part-00003-3ec73052-68ac-445c-803f-c54b87e9f16d-c000.snappy.parquet.crc
   ... (13 archivos más)

════════════════════════════════════════════════════════════
📂 PARQUET PARTICIONADO (por anio / mes):
════════════════════════════════════════════════════════════
📁 ventas_parquet_por_anio_mes/
   ✅ ._SUCCESS.crc
   ✅ _SUCCESS
   📁 anio=2021/
      📁 mes=1/
         📎 .part-00000-60042b29-8109-4fe2-972d-69420ece4acc.c000.snappy.parquet.crc
         📎 .part-00001-60042b29-8109-4fe2-972d-69420ece4acc.c000.snappy.parquet.crc
         ... (14 archivos más)
     

## Paso 9 — Comparación de tamaños en disco

Con un dataset real de ~52 MB en Excel, la diferencia entre CSV y Parquet es **muy visible**. Parquet usa compresión Snappy por defecto.

In [16]:
def size_mb(path):
    """Calcula el tamaño total de una carpeta en MB."""
    total = 0
    if os.path.exists(path):
        for root, _, files in os.walk(path):
            for f in files:
                if not f.startswith('.'):
                    total += os.path.getsize(os.path.join(root, f))
    return total / (1024 * 1024)

def count_parquet_files(path):
    """Cuenta archivos .parquet en una carpeta."""
    total = 0
    if os.path.exists(path):
        for root, _, files in os.walk(path):
            total += sum(1 for f in files if f.endswith('.parquet'))
    return total

sz_csv      = size_mb(RUTA_CSV)
sz_parquet  = size_mb(RUTA_PARQUET_SIMPLE)
sz_part     = size_mb(RUTA_PARQUET_PART_MES)

print("📊 COMPARACIÓN DE TAMAÑOS EN DISCO:")
print("═" * 55)
print(f"  {'Estrategia':<35} {'Tamaño':>8}  {'Archivos'}")
print("  " + "-" * 53)

n_csv = len([f for r, _, fs in os.walk(RUTA_CSV) if os.path.exists(RUTA_CSV) for f in fs if f.endswith('.csv')])
n_pq  = count_parquet_files(RUTA_PARQUET_SIMPLE)
n_pqp = count_parquet_files(RUTA_PARQUET_PART_MES)

print(f"  {'CSV (texto plano)':<35} {sz_csv:>6.2f} MB  {n_csv} archivos")
print(f"  {'Parquet simple (sin partición)':<35} {sz_parquet:>6.2f} MB  {n_pq} archivos")
print(f"  {'Parquet particionado (anio/mes)':<35} {sz_part:>6.2f} MB  {n_pqp} archivos")
print("═" * 55)

if sz_parquet > 0 and sz_csv > 0:
    reduccion = (1 - sz_parquet / sz_csv) * 100
    print(f"\n  ✅ Parquet es {sz_csv/sz_parquet:.1f}x más pequeño que CSV")
    print(f"  ✅ Ahorra un {reduccion:.0f}% de espacio en disco")

print()
print("💡 Nota: El Parquet particionado puede ser similar o mayor en tamaño")
print("   que el simple porque crea múltiples archivos pequeños (uno por partición).")
print("   Su ventaja NO es el tamaño sino la VELOCIDAD DE CONSULTA por filtrado.")

📊 COMPARACIÓN DE TAMAÑOS EN DISCO:
═══════════════════════════════════════════════════════
  Estrategia                            Tamaño  Archivos
  -----------------------------------------------------
  CSV (texto plano)                   122.78 MB  8 archivos
  Parquet simple (sin partición)       10.80 MB  8 archivos
  Parquet particionado (anio/mes)      21.21 MB  511 archivos
═══════════════════════════════════════════════════════

  ✅ Parquet es 11.4x más pequeño que CSV
  ✅ Ahorra un 91% de espacio en disco

💡 Nota: El Parquet particionado puede ser similar o mayor en tamaño
   que el simple porque crea múltiples archivos pequeños (uno por partición).
   Su ventaja NO es el tamaño sino la VELOCIDAD DE CONSULTA por filtrado.


## Paso 10 — Leer el Parquet particionado

Leemos el Parquet particionado y verificamos que los datos estén completos y los tipos de datos preservados.

In [17]:
# ── Leer el Parquet particionado ──
print("⏳ Leyendo Parquet particionado...")
df_parquet = spark.read.parquet(RUTA_PARQUET_PART_MES)

print(f"✅ Parquet particionado leído correctamente")
print(f"   Registros recuperados : {df_parquet.count():,}")
print(f"   Columnas              : {len(df_parquet.columns)}")
print()

print("📋 Schema del Parquet (los tipos se preservaron):")
df_parquet.printSchema()

# Vista previa
cols_show = [c for c in [FECHA_COL, TOTAL_COL, "anio", "mes"] if c in df_parquet.columns]
print("📄 Primeras 5 filas recuperadas:")
df_parquet.select(*cols_show).show(5)

⏳ Leyendo Parquet particionado...
✅ Parquet particionado leído correctamente


[Stage 47:>                                                       (0 + 27) / 27]

   Registros recuperados : 522,688
   Columnas              : 33

📋 Schema del Parquet (los tipos se preservaron):
root
 |-- fecha_de_emision: date (nullable = true)
 |-- tipo: string (nullable = true)
 |-- serie: string (nullable = true)
 |-- numero: string (nullable = true)
 |-- doc_entidad_tipo: string (nullable = true)
 |-- doc_entidad_numero: string (nullable = true)
 |-- denominacion_entidad: string (nullable = true)
 |-- moneda: string (nullable = true)
 |-- tipo_de_cambio: string (nullable = true)
 |-- unidad_de_medida: string (nullable = true)
 |-- codigo: string (nullable = true)
 |-- descripcion: string (nullable = true)
 |-- cantidad: string (nullable = true)
 |-- valor_unitario: string (nullable = true)
 |-- precio_unitario: string (nullable = true)
 |-- descuento: string (nullable = true)
 |-- subtotal: string (nullable = true)
 |-- tipo_de_igv: string (nullable = true)
 |-- igv: string (nullable = true)
 |-- tipo_de_isc: double (nullable = true)
 |-- isc: string (nullabl

## Paso 11 — Partition Pruning: Spark lee solo lo que necesitas

**Partition Pruning** = poda de particiones. Cuando filtras por una columna de partición, Spark **skip** (salta) todas las carpetas que no contienen datos relevantes.

```
Consulta: "Dame ventas de enero 2023"

Sin partición:        Con partición anio/mes:
└── part-00000.parq  ├── anio=2021/ ← SALTADA ✅
    (lee TODO)        ├── anio=2022/ ← SALTADA ✅
                      ├── anio=2023/
                      │   ├── mes=1/ ← LEÍDA 📖
                      │   ├── mes=2/ ← SALTADA ✅
                      │   └── ...
                      └── anio=2024/ ← SALTADA ✅
```

In [18]:
# ── Consulta con Partition Pruning ──
# Elige un año que exista en tu dataset
ANIO_CONSULTA = 2023
MES_CONSULTA  = 1  # Enero

print(f"📊 Consulta con Partition Pruning: ventas de {ANIO_CONSULTA}-{MES_CONSULTA:02d}")
print(f"   Spark leerá SOLO la carpeta: anio={ANIO_CONSULTA}/mes={MES_CONSULTA}/")
print()

df_consulta = df_parquet.filter(
    (col("anio") == ANIO_CONSULTA) & (col("mes") == MES_CONSULTA)
)

n_consulta = df_consulta.count()

if TOTAL_COL in df_consulta.columns:
    resultado = df_consulta.agg(
        count("*").alias("num_registros"),
        _sum(TOTAL_COL).alias("total_ventas"),
        avg(TOTAL_COL).alias("promedio_venta")
    ).collect()[0]

    print(f"   Resultados de {ANIO_CONSULTA}-{MES_CONSULTA:02d}:")
    print(f"   Registros     : {resultado['num_registros']:,}")
    print(f"   Total ventas  : S/ {resultado['total_ventas']:,.2f}")
    print(f"   Promedio      : S/ {resultado['promedio_venta']:,.2f}")

# Ver el plan de ejecución — busca "PartitionFilters" en el Physical Plan
print()
print("🔍 Plan de ejecución (observa 'PartitionFilters' — evidencia del pruning):")
print("=" * 60)
df_consulta.explain(False)

📊 Consulta con Partition Pruning: ventas de 2023-01
   Spark leerá SOLO la carpeta: anio=2023/mes=1/

   Resultados de 2023-01:
   Registros     : 1,096
   Total ventas  : S/ 5,099,930.28
   Promedio      : S/ 4,653.22

🔍 Plan de ejecución (observa 'PartitionFilters' — evidencia del pruning):
== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [fecha_de_emision#755,tipo#756,serie#757,numero#758,doc_entidad_tipo#759,doc_entidad_numero#760,denominacion_entidad#761,moneda#762,tipo_de_cambio#763,unidad_de_medida#764,codigo#765,descripcion#766,cantidad#767,valor_unitario#768,precio_unitario#769,descuento#770,subtotal#771,tipo_de_igv#772,igv#773,tipo_de_isc#774,isc#775,impuesto_bolsas#776,total#777,anulado#778,FECHA_DT#779,... 8 more fields] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/opt/artifacts/nb04_almacenamiento/ventas_parquet_por_anio_mes], PartitionFilters: [isnotnull(anio#786), isnotnull(mes#787), (anio#786 = 2023), (mes#787 = 1

In [19]:
# ── Consultas analíticas adicionales desde el Parquet particionado ──

# Pregunta 1: Top 3 meses con más ventas
if TOTAL_COL in df_parquet.columns and "anio_mes" in df_parquet.columns:
    print("🏆 Top 5 meses con mayor volumen de ventas:")
    df_parquet.groupBy("anio_mes").agg(
        _sum(TOTAL_COL).alias("total_ventas"),
        count("*").alias("num_registros")
    ).filter(col("anio_mes").isNotNull()) \
     .orderBy(F.desc("total_ventas")) \
     .limit(5) \
     .show(truncate=False)

# Pregunta 2: Ventas por año
if TOTAL_COL in df_parquet.columns and "anio" in df_parquet.columns:
    print("📊 Ventas totales por año:")
    df_parquet.groupBy("anio").agg(
        count("*").alias("num_registros"),
        _sum(TOTAL_COL).alias("total_ventas_soles")
    ).filter(col("anio").isNotNull()) \
     .orderBy("anio") \
     .show(truncate=False)

# Pregunta 3: Ventas por trimestre del último año
if TOTAL_COL in df_parquet.columns and "trimestre" in df_parquet.columns:
    anio_max = df_parquet.filter(col("anio").isNotNull()).agg(_max("anio")).collect()[0][0]
    if anio_max:
        print(f"📊 Ventas por trimestre en {anio_max}:")
        df_parquet.filter(col("anio") == anio_max).groupBy("trimestre").agg(
            count("*").alias("num_registros"),
            _sum(TOTAL_COL).alias("total_ventas_soles")
        ).orderBy("trimestre").show(truncate=False)

🏆 Top 5 meses con mayor volumen de ventas:


+--------+---------------------+-------------+
|anio_mes|total_ventas         |num_registros|
+--------+---------------------+-------------+
|2025-12 |1.1390354208664492E10|4441         |
|2025-11 |1.0350395173861858E10|4594         |
|2026-02 |5.292219412706148E9  |2172         |
|2025-10 |4.939095725425035E9  |2957         |
|2026-01 |4.6544665599141035E9 |1987         |
+--------+---------------------+-------------+

📊 Ventas totales por año:


+----+-------------+---------------------+
|anio|num_registros|total_ventas_soles   |
+----+-------------+---------------------+
|2021|126411       |7.621456274403249E7  |
|2022|122011       |1.6730820717714074E8 |
|2023|98207        |5.184849068916412E9  |
|2024|98740        |2.468539124537337E9  |
|2025|72328        |2.9079567844475426E10|
|2026|4991         |1.0252001539884804E10|
+----+-------------+---------------------+



📊 Ventas por trimestre en 2026:
+---------+-------------+---------------------+
|trimestre|num_registros|total_ventas_soles   |
+---------+-------------+---------------------+
|1        |4944         |1.0249911652951944E10|
|2        |47           |2089886.9328599987   |
+---------+-------------+---------------------+



## Paso 12 — Lectura selectiva columnar: la ventaja invisible de Parquet

Una de las mayores ventajas de Parquet es que cuando haces `.select()` de pocas columnas, Spark **solo lee esas columnas del disco** — no lee todo el registro.

En CSV: leer 2 columnas = leer **todo el archivo** igualmente.  
En Parquet: leer 2 columnas = leer **solo esos 2 bloques** de datos.

In [20]:
# ── Demostración de lectura selectiva columnar ──
print("🔍 Lectura selectiva: solo 3 columnas desde Parquet")
print("   (Spark accede solo a los bloques de esas columnas en disco)")
print()

# Seleccionamos solo fecha, anio_mes y total — Parquet lee solo esos 3 bloques
cols_select = [c for c in [FECHA_COL, "anio_mes", TOTAL_COL] if c in df_parquet.columns]
df_parquet.select(*cols_select).show(8)

# Para comparar: leer el CSV necesita cargar todas las columnas
print()
df_csv = spark.read.option("header", True).csv(RUTA_CSV)
print(f"📂 CSV cargado: {df_csv.count():,} registros | {len(df_csv.columns)} columnas")
print("   Aunque seleccionemos pocas columnas, CSV cargó TODAS igualmente.")

print()
print("📌 Esto es por qué Parquet es más eficiente en analítica:")
print("   Si tienes 50 columnas pero solo necesitas 3, CSV lee 50, Parquet lee solo 3.")

🔍 Lectura selectiva: solo 3 columnas desde Parquet
   (Spark accede solo a los bloques de esas columnas en disco)

+----------------+--------+---------------+
|fecha_de_emision|anio_mes|total_acumulado|
+----------------+--------+---------------+
|      2023-10-06| 2023-10|     1423.17365|
|      2023-10-06| 2023-10|     1423.17365|
|      2023-10-06| 2023-10|     1423.17365|
|      2023-10-06| 2023-10|     1423.17365|
|      2023-10-06| 2023-10|     1423.17365|
|      2023-10-06| 2023-10|     1423.17365|
|      2023-10-06| 2023-10|     1423.17365|
|      2023-10-06| 2023-10|     1423.17365|
+----------------+--------+---------------+
only showing top 8 rows

📂 CSV cargado: 522,688 registros | 33 columnas
   Aunque seleccionemos pocas columnas, CSV cargó TODAS igualmente.

📌 Esto es por qué Parquet es más eficiente en analítica:
   Si tienes 50 columnas pero solo necesitas 3, CSV lee 50, Parquet lee solo 3.


## Paso 13 — Reparticion y coalesce: controlar el número de archivos

Cuando Spark escribe un Parquet, crea **un archivo por partición del DataFrame**. Si tienes 8 particiones → 8 archivos por carpeta de partición.

- **`coalesce(1)`** → fuerza a escribir 1 solo archivo por carpeta (menos archivos, pero puede ser lento)
- **`repartition(n)`** → redistribuye en n particiones (puede haber shuffle)

Para nuestro proyecto usaremos `coalesce(1)` para tener archivos más limpios y fáciles de inspeccionar.

In [21]:
# ── Guardar Parquet particionado con 1 archivo por partición ──
RUTA_PARQUET_COALESCE = f"{RUTA_BASE}/ventas_parquet_final"

print(f"💾 Guardando Parquet final (coalesce=1, partición por anio/mes)...")
print(f"   Esto crea exactamente 1 archivo por carpeta anio=X/mes=Y/")

(
    df_preparado
    .coalesce(1)                   # 1 archivo por partición lógica de Spark
    .write
    .mode("overwrite")
    .partitionBy("anio", "mes")    # subcarpetas por año y mes
    .parquet(RUTA_PARQUET_COALESCE)
)

n_archivos_final = count_parquet_files(RUTA_PARQUET_COALESCE)
sz_final = size_mb(RUTA_PARQUET_COALESCE)

print(f"\n✅ Parquet final guardado:")
print(f"   Archivos generados : {n_archivos_final}")
print(f"   Tamaño total       : {sz_final:.2f} MB")
print(f"   Ruta               : {RUTA_PARQUET_COALESCE}")
print()

# Mostrar estructura
print("📁 Estructura física final:")
mostrar_estructura(RUTA_PARQUET_COALESCE, max_niveles=3, max_archivos=2)

💾 Guardando Parquet final (coalesce=1, partición por anio/mes)...
   Esto crea exactamente 1 archivo por carpeta anio=X/mes=Y/



✅ Parquet final guardado:
   Archivos generados : 64
   Tamaño total       : 15.08 MB
   Ruta               : ../artifacts/nb04_almacenamiento/ventas_parquet_final

📁 Estructura física final:
📁 ventas_parquet_final/
   ✅ ._SUCCESS.crc
   ✅ _SUCCESS
   📁 anio=2021/
      📁 mes=1/
         📎 .part-00000-2c390425-aa81-468f-9926-a70fb63e6457.c000.snappy.parquet.crc
         📄 part-00000-2c390425-aa81-468f-9926-a70fb63e6457.c000.snappy.parquet
      📁 mes=10/
         📎 .part-00000-2c390425-aa81-468f-9926-a70fb63e6457.c000.snappy.parquet.crc
         📄 part-00000-2c390425-aa81-468f-9926-a70fb63e6457.c000.snappy.parquet
      📁 mes=11/
         📎 .part-00000-2c390425-aa81-468f-9926-a70fb63e6457.c000.snappy.parquet.crc
         📄 part-00000-2c390425-aa81-468f-9926-a70fb63e6457.c000.snappy.parquet
      📁 mes=12/
         📎 .part-00000-2c390425-aa81-468f-9926-a70fb63e6457.c000.snappy.parquet.crc
         📄 part-00000-2c390425-aa81-468f-9926-a70fb63e6457.c000.snappy.parquet
      📁 mes=2/
    

## Paso 14 — Verificación final del pipeline completo

Cerramos el ciclo: **leer el Parquet final** y ejecutar las consultas que se benefician del particionado.

In [22]:
# ── Leer el Parquet final y verificar ──
df_final = spark.read.parquet(RUTA_PARQUET_COALESCE)

print("✅ VERIFICACIÓN FINAL DEL PIPELINE DE ALMACENAMIENTO:")
print("═" * 55)
print(f"   Registros en salida final  : {df_final.count():,}")
print(f"   Columnas                   : {len(df_final.columns)}")

if FECHA_COL in df_final.columns:
    rango_f = df_final.agg(
        _min(FECHA_COL).alias("desde"),
        _max(FECHA_COL).alias("hasta")
    ).collect()[0]
    print(f"   Rango temporal             : {rango_f['desde']} → {rango_f['hasta']}")

if TOTAL_COL in df_final.columns:
    total_f = df_final.agg(_sum(TOTAL_COL)).collect()[0][0]
    print(f"   Total ventas (S/)          : {total_f:,.2f}")

print()
print("📄 Muestra del Parquet final:")
cols_verif = [c for c in [FECHA_COL, TOTAL_COL, "anio", "mes", "trimestre"] if c in df_final.columns]
df_final.select(*cols_verif).show(8)

✅ VERIFICACIÓN FINAL DEL PIPELINE DE ALMACENAMIENTO:
═══════════════════════════════════════════════════════
   Registros en salida final  : 522,688
   Columnas                   : 33
   Rango temporal             : 2021-01-25 → 2026-04-06
   Total ventas (S/)          : 47,228,480,347.74

📄 Muestra del Parquet final:
+----------------+-----------------+----+---+---------+
|fecha_de_emision|  total_acumulado|anio|mes|trimestre|
+----------------+-----------------+----+---+---------+
|      2021-03-22|             13.0|2021|  3|        1|
|      2021-03-08|         73.83333|2021|  3|        1|
|      2021-03-08|         73.83333|2021|  3|        1|
|      2021-03-08|         73.83333|2021|  3|        1|
|      2021-03-23|84.66666000000001|2021|  3|        1|
|      2021-03-23|84.66666000000001|2021|  3|        1|
|      2021-03-30|97.66666000000001|2021|  3|        1|
|      2021-03-23|         52.23338|2021|  3|        1|
+----------------+-----------------+----+---+---------+
only sho

In [23]:
# ── Consulta analítica final: top 3 mejores meses historicos ──
print("🏆 Top 3 mejores meses históricos (consultados desde Parquet particionado):")
print("   (Spark usa Partition Pruning automáticamente por anio y mes)")
print()

if TOTAL_COL in df_final.columns and "anio_mes" in df_final.columns:
    df_final.groupBy("anio_mes", "anio", "mes").agg(
        count("*").alias("num_registros"),
        _sum(TOTAL_COL).alias("total_ventas"),
        avg(TOTAL_COL).alias("promedio_venta")
    ).filter(col("anio_mes").isNotNull()) \
     .orderBy(F.desc("total_ventas")) \
     .limit(3) \
     .show(truncate=False)
else:
    print("   (Columnas anio_mes o total no disponibles en el Parquet)")

🏆 Top 3 mejores meses históricos (consultados desde Parquet particionado):
   (Spark usa Partition Pruning automáticamente por anio y mes)

+--------+----+---+-------------+---------------------+-----------------+
|anio_mes|anio|mes|num_registros|total_ventas         |promedio_venta   |
+--------+----+---+-------------+---------------------+-----------------+
|2025-12 |2025|12 |4441         |1.1390354208664507E10|2564817.430458119|
|2025-11 |2025|11 |4594         |1.0350395173861874E10|2253024.63514625 |
|2026-02 |2026|2  |2172         |5.292219412706117E9  |2436565.107139096|
+--------+----+---+-------------+---------------------+-----------------+



## Paso 15 — Ficha técnica de diseño: justificación del almacenamiento

Como ingenieros de datos, documentamos las decisiones técnicas tomadas.

In [24]:
# ── Resumen final de comparación de estrategias ──
sz_csv_final  = size_mb(RUTA_CSV)
sz_pq_final   = size_mb(RUTA_PARQUET_SIMPLE)
sz_pqp_final  = size_mb(RUTA_PARQUET_COALESCE)

print("═" * 65)
print("📊 TABLA COMPARATIVA FINAL DE ESTRATEGIAS DE ALMACENAMIENTO")
print("═" * 65)
print(f"  {'Criterio':<30} {'CSV':>8} {'Parquet':>8} {'Parq+Part':>10}")
print("  " + "-" * 60)
print(f"  {'Compresión automática':<30} {'❌ No':>8} {'✅ Sí':>8} {'✅ Sí':>10}")
print(f"  {'Tipos de datos preservados':<30} {'❌ No':>8} {'✅ Sí':>8} {'✅ Sí':>10}")
print(f"  {'Lectura columnar':<30} {'❌ No':>8} {'✅ Sí':>8} {'✅ Sí':>10}")
print(f"  {'Partition Pruning':<30} {'❌ No':>8} {'❌ No':>8} {'✅ Sí':>10}")
print(f"  {'Velocidad de consulta temporal':<30} {'🔴 Lenta':>8} {'🟡 Media':>8} {'🟢 Rápida':>10}")
print(f"  {'Tamaño en disco':<30} {sz_csv_final:>6.2f} MB {sz_pq_final:>5.2f} MB {sz_pqp_final:>7.2f} MB")
print(f"  {'Archivos generados':<30} {'N':>8} {'N':>8} {'1/partición':>10}")
print("═" * 65)

print()
print("✅ DISEÑO ELEGIDO PARA EL PROYECTO:")
print("   Formato    : Parquet con compresión Snappy")
print("   Partición  : anio / mes")
print("   Coalesce   : 1 (1 archivo por partición)")
print()
print("📌 JUSTIFICACIÓN:")
print("   Las consultas del negocio filtran frecuentemente por PERIODO")
print("   (mes, trimestre, año). El particionado por anio/mes permite")
print("   que Spark salte años/meses no relevantes (Partition Pruning).")
print()
print("⚠️  RIESGO DEL DISEÑO:")
print("   Si el dataset crece mucho en años (ej: 20 años de datos), la")
print("   cantidad de carpetas podría hacerse difícil de administrar.")
print("   Alternativa: particionar solo por 'anio' si 'mes' no se usa")
print("   frecuentemente como filtro.")

═════════════════════════════════════════════════════════════════
📊 TABLA COMPARATIVA FINAL DE ESTRATEGIAS DE ALMACENAMIENTO
═════════════════════════════════════════════════════════════════
  Criterio                            CSV  Parquet  Parq+Part
  ------------------------------------------------------------
  Compresión automática              ❌ No     ✅ Sí       ✅ Sí
  Tipos de datos preservados         ❌ No     ✅ Sí       ✅ Sí
  Lectura columnar                   ❌ No     ✅ Sí       ✅ Sí
  Partition Pruning                  ❌ No     ❌ No       ✅ Sí
  Velocidad de consulta temporal  🔴 Lenta  🟡 Media   🟢 Rápida
  Tamaño en disco                122.78 MB 10.80 MB   15.08 MB
  Archivos generados                    N        N 1/partición
═════════════════════════════════════════════════════════════════

✅ DISEÑO ELEGIDO PARA EL PROYECTO:
   Formato    : Parquet con compresión Snappy
   Partición  : anio / mes
   Coalesce   : 1 (1 archivo por partición)

📌 JUSTIFICACIÓN:
   Las cons

## Paso 16 — Cierre: Resumen de la sesión

### ¿Qué aprendimos?

| Concepto | Descripción | En nuestro proyecto |
|---|---|---|
| **HDFS** | Sistema de archivos distribuido | Simulado con sistema de archivos local |
| **Zonas de datos** | Raw → Processed → Analytics | Excel → Parquet limpio → Parquet particionado |
| **Formato Parquet** | Columnar, comprimido, con schema | Usado para toda la salida del pipeline |
| **partitionBy()** | Divide datos en subcarpetas | Por `anio` y `mes` |
| **Partition Pruning** | Spark lee solo las carpetas necesarias | Filtros por `anio` y `mes` son muy rápidos |
| **Lectura columnar** | Solo lee las columnas que usas | `.select()` en Parquet es muy eficiente |
| **coalesce(1)** | 1 archivo por partición | Estructura limpia y manejable |

### Preguntas para reflexión:
1. ¿Por qué NO conviene particionar por `cliente_id` en este dataset?
2. ¿Qué ventaja tiene leer el Parquet del NB03 en lugar de volver a leer el Excel?
3. ¿En qué caso el CSV sería mejor que Parquet?
4. ¿Qué diferencia conceptual hay entre "zona de datos" y "formato de salida"?

---

### Resumen del pipeline ejecutado:

| Paso | Tarea | Estado |
|---|---|---|
| 1 | SparkSession configurada | ✅ |
| 2 | Leer desde zona Processed (Parquet NB03) o Raw (Excel) | ✅ |
| 3 | Explorar estructura y cobertura temporal | ✅ |
| 4 | Preparar columnas de partición (anio, mes, trimestre) | ✅ |
| 5 | Validación de calidad antes de almacenar | ✅ |
| 6 | Decisión de diseño: formato + particionado | ✅ |
| 7 | Guardar CSV, Parquet simple y Parquet particionado | ✅ |
| 8 | Explorar estructura física de carpetas | ✅ |
| 9 | Comparar tamaños en disco | ✅ |
| 10 | Leer Parquet particionado y verificar | ✅ |
| 11 | Demostrar Partition Pruning con explain() | ✅ |
| 12 | Lectura columnar selectiva | ✅ |
| 13 | Guardar versión final con coalesce(1) | ✅ |
| 14 | Consultas analíticas finales | ✅ |
| 15 | Ficha técnica de diseño justificada | ✅ |